# MMCultureQA (SemEval 2027, Task 2): the official baseline, on every released split

This notebook clones the project repository and runs the organisers' reference system
(**Qwen2.5-Omni-3B, 4-bit NF4, greedy, 64 new tokens, 512·28² pixels, their prompt**) with
its settings unchanged. Then it scores the output with the leaderboard's own scorer and
writes every table, chart and log the report needs.

| stage | split | items | why |
|---|---|---|---|
| `dev` | dev | 4 × 1,000 | main result (has references) |
| `devtest` | devtest | 4 × 1,000 | leaderboard submission files (no references) |
| `train_sample` | train | 4 × 250, seeded | generalisation check (40k full train ≈ 100 T4-hours) |
| `dev_text_only` | dev | 4 × 250 | ablation: the same call with the image removed |

**Before you run:** Settings → Accelerator → **GPU T4 x2**, Internet **on**. Then use
**Save Version → Save & Run All (Commit)**. A commit keeps running with the browser closed.
The run stops starting new items after `BUDGET_HOURS`, so it always finishes inside
Kaggle's 12-hour limit, and anything not reached is listed in `logs/completion.json`.

When it finishes, download **`mmcqa_baseline_results.zip`** from the Output tab.

## 1. Config

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-baseline.git"   # <- your copy of this repository
BRANCH = "main"
STAGES = None          # None = all four stages; or e.g. ["dev"] / ["dev", "devtest"]
LIMIT = None           # e.g. 4 = smoke test (4 items per track and stage, ~10 min); None = full run
BUDGET_HOURS = 10.5    # stop starting new generations after this; scoring + analysis need ~20 min more

WORK = "/kaggle/working/results"
CODE = "/kaggle/working/mmcqa-baseline"

## 2. Code and dependencies
Clones the repository. A Kaggle dataset holding the repo, attached under `/kaggle/input`, also works if GitHub is unreachable.

In [ ]:
import glob, os, subprocess, sys
if not os.path.isdir(CODE):
    local = glob.glob("/kaggle/input/**/mmcqa_baseline/__init__.py", recursive=True)
    if local:
        subprocess.run(["cp", "-r", os.path.dirname(os.path.dirname(local[0])), CODE], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!pip install -q -r {CODE}/requirements.txt
!nvidia-smi -L

## 3. Inference: the official baseline on every stage
Data and the starter kit are fetched at pinned revisions. Two worker processes per GPU each answer a share of the plan, and every answer is written to disk as soon as it is produced. Progress is printed every minute, and per-worker logs are in `results/logs/`.

In [ ]:
args = f"--workdir {WORK} --budget-hours {BUDGET_HOURS}"
if STAGES: args += " --stages " + " ".join(STAGES)
if LIMIT: args += f" --limit {LIMIT}"
!cd {CODE} && python -m mmcqa_baseline.run {args}

## 4. Score with the official scorer
`backbone.evaluate` from the organisers' scorer: mmBERT layer 17 at the pinned revision, BLEU (flores200), ROUGE-L and coverage. mBERT is added for comparison only.

In [ ]:
!cd {CODE} && python -m mmcqa_baseline.score --workdir {WORK}

## 5. Analysis: tables, charts, examples

In [ ]:
!cd {CODE} && python -m mmcqa_baseline.analysis --workdir {WORK}

## 5b. Second-level analysis
Dataset statistics, train overlap, answer anatomy, post-processing probes re-scored with the official scorer, metric calibration, dialect identification (CAMeLBERT MADAR-6), vision ablation by question type, statistical power. No GPU needed (about 45 min on CPU, less on GPU).

In [ ]:
!cd {CODE} && python -m mmcqa_baseline.insights --workdir {WORK}

## 5c. LLM judge (optional, the task paper's second evaluation)
Runs only if a Kaggle secret `OPENROUTER_API_KEY` exists. 250 dev items per track × 4 systems with Gemini 2.5 Flash, about $1.70. It is not part of the leaderboard.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    print("no OPENROUTER_API_KEY secret: skipping the judge")
if os.environ.get("OPENROUTER_API_KEY"):
    !cd {CODE} && python -m mmcqa_baseline.judge --workdir {WORK} --per-track 250

In [ ]:
from IPython.display import Markdown, Image, display
for name in ("summary.md", "insights.md"):
    if os.path.exists(f"{WORK}/report/{name}"):
        display(Markdown(open(f"{WORK}/report/{name}", encoding="utf-8").read()))
for png in sorted(glob.glob(f"{WORK}/report/charts/*.png")):
    print(os.path.basename(png)); display(Image(png, width=650))

## 6. Package the results
Everything except the images and the starter-kit clone (both re-downloadable) goes into one zip.

In [ ]:
import shutil
for tmp in ("media", "mmcultureqa"):
    shutil.rmtree(f"{WORK}/{tmp}", ignore_errors=True)
!cd /kaggle/working && zip -qr mmcqa_baseline_results.zip results && ls -lh mmcqa_baseline_results.zip
print("devtest submission files:", sorted(glob.glob(f"{WORK}/predictions/devtest/*.zip")))